# Qwen3-0.6B LoRA/QLoRA — Colab starter

এই Notebook Android ফোনের browser থেকেও Google Colab-এ চালানো যাবে। **Runtime → Change runtime type → T4 GPU** (বা উপলব্ধ GPU) নির্বাচন করুন।

এটি `Qwen/Qwen3-0.6B`-এর উপর LoRA/QLoRA adapter train করে। এটি base model-এর built-in safety alignment মুছে ফেলে না; benign প্রশ্নে অযথা refusal কমানো এবং নিজের বৈধ use-case অনুযায়ী assistant behavior শেখানোর জন্য reviewed examples ব্যবহার করুন।

In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
else: print('GPU পাওয়া যায়নি। Runtime > Change runtime type > GPU নির্বাচন করুন।')

## 1. Training code GitHub থেকে আনুন

In [ ]:
!git clone --depth 1 --branch feature/colab-finetuning-starter https://github.com/mdarman0964/Qwen3.git /content/Qwen3
%cd /content/Qwen3
!ls -la training

## 2. প্রয়োজনীয় package ইনস্টল করুন

In [ ]:
%pip -q install -r training/requirements.txt

## 3. Dataset যাচাই করুন

`training/data/sample.jsonl`-এ কয়েকটি format example আছে। এটি শুধু pipeline smoke test-এর জন্য, production training-এর জন্য যথেষ্ট নয়। নিজের যাচাইকৃত উদাহরণ যোগ করতে `training/data/train.jsonl` তৈরি করে নিচের command-এ path বদলান।

In [ ]:
!python - <<'PY'
import json
p='training/data/sample.jsonl'
rows=[json.loads(x) for x in open(p, encoding='utf-8') if x.strip()]
print('Examples:', len(rows))
print('First conversation:', rows[0]['messages'])
PY

## 4. ছোট smoke-test training চালান

নিচের `--max_steps 20` কেবল pipeline যাচাইয়ের জন্য। ভালো ফলাফলের জন্য বড়, পরিষ্কার dataset ব্যবহার করুন এবং `--max_steps` বাদ দিয়ে epochs অনুযায়ী train করুন। GPU memory কম হলে `--max_length 512` রাখুন।

In [ ]:
!python training/train.py \
  --model_id Qwen/Qwen3-0.6B \
  --data_path training/data/sample.jsonl \
  --output_dir /content/qwen3-0.6b-lora \
  --max_length 512 \
  --batch_size 1 \
  --gradient_accumulation_steps 4 \
  --max_steps 20

## 5. Adapter ZIP করে ডাউনলোড করুন

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/qwen3-0.6b-lora', 'zip', '/content/qwen3-0.6b-lora')
print(zip_path)
files.download(zip_path)

## Troubleshooting ও safety checks

- CUDA out of memory: batch size 1, max length 512 বা 256 রাখুন; প্রয়োজনে gradient accumulation কমান।
- প্রথমে sample dataset দিয়ে শুধু pipeline যাচাই করুন। বাস্তব ফলের জন্য বহু মানসম্মত এবং reviewed example দরকার।
- Training set-এ benign request-এর পাশাপাশি যথাযথ refusal/redirect example রাখুন, যাতে false refusal কমালেও প্রয়োজনীয় safety boundary বজায় থাকে।
- আলাদা held-out test set-এ ফল যাচাই করুন।
- Password, API token, private messages বা sensitive personal data dataset-এ রাখবেন না।
- Adapter base model নয়; inference-এ base model এবং adapter দুটোই লাগবে।